# Revisão do M7

Este notebook audita o indicador de omissão de planejamento da RQ3.

**Conclusão:** M7 reproduz corretamente a ausência de nota M6, mas não identifica omissão de planejamento. A saída útil é uma trajetória de inatividade recente no repositório, calculada em janelas móveis de sete dias antes de cada checkpoint de avaliação; ela distingue início tardio de inatividade persistente.

## Estrutura recomendada

| Saída | Pergunta respondida | Relação com M6/M3/M4 |
|---|---|---|
| **M7a - Trajetória de inatividade recente** | Qual fração das equipes não teve commit nos 7 dias anteriores a cada checkpoint? | Dinâmica binária de atividade; não mede autoria (M3) nem magnitude (M4) |
| **M7b - Inatividade persistente** | Quais equipes permanecem inativas em checkpoints consecutivos? | Distingue atraso inicial de ausência sustentada |
| **M7c - Cobertura cruzada de evidência** | Equipes inativas no Git foram observadas por avaliadores? | Demonstra os limites do Git; não é score de planejamento |

M7 não deve duplicar M6a como preditor em M9, nem ser interpretado como ausência de planejamento fora do repositório.

## 1. Vínculo com a RQ

O notebook extrai a RQ de M7 diretamente do texto atual do paper e falha se M7 não estiver na subseção RQ3.

In [ ]:
from hashlib import sha256
from pathlib import Path
import re

import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'paper_v8/latex_code/main.tex').is_file():
            return candidate
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = find_project_root(Path.cwd())
PAPER_PATH = PROJECT_ROOT / 'paper_v8/latex_code/main.tex'
M7_PATH = PROJECT_ROOT / 'paper_v8/data/m7_planning_omission_rate.csv'
SIGNALS_PATH = PROJECT_ROOT / 'paper_v4/advanced_metrics/outputs/team_level_signals.csv'
PLANNING_PATH = PROJECT_ROOT / 'data/analysis/planning_metrics.parquet'
COMMITS_PATH = PROJECT_ROOT / 'data/lake/git_commits.parquet'
EVALUATOR_PATH = PROJECT_ROOT / 'data/lake/evaluator_team_cuts.parquet'

paper_text = PAPER_PATH.read_text(encoding='utf-8')
rq_matches = dict(re.findall(r'\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n', paper_text))
m7_position = paper_text.index(r'\textbf{M7 --')
rq3_position = paper_text.index(r'\subsubsection{RQ3:')
detected_rq = 'RQ3' if rq3_position < m7_position else 'UNKNOWN'
assert detected_rq == 'RQ3'
assert rq_matches[detected_rq]
pd.DataFrame([{'metric':'M7','detected_rq':detected_rq,'rq_text':rq_matches[detected_rq],'paper_sha256':sha256(paper_text.encode()).hexdigest()}])

In [ ]:
CONFIG = {
    'metric': 'M7',
    'expected_rq': 'RQ3',
    'unit_of_analysis': 'team_semester_rolling_window_then_cohort',
    'primary_outputs': [
        'rolling_7d_repository_inactivity_trajectory',
        'checkpoint_7d_repository_inactivity_rate',
        'persistent_repository_inactivity_pattern',
        't1_evaluator_coverage_among_inactive',
    ],
    'rolling_window_days': 7,
    'rolling_step_days': 1,
    'rolling_end_day_range_relative_to_t3': [-63, 7],
    'checkpoint_anchor': 'last_evaluator_vote_per_team_marker',
    'rolling_alignment_anchor': 'last_evaluator_vote_per_team_t3',
    'planning_claim': 'not_identifiable_from_repository_inactivity',
    'relationship_to_m6a': 'derived_secondary_diagnostic_only',
    'inference': 'descriptive_only',
}
assert CONFIG['expected_rq'] == detected_rq
assert CONFIG['rolling_window_days'] == 7
assert CONFIG['relationship_to_m6a'] == 'derived_secondary_diagnostic_only'
CONFIG

## 2. Auditoria do M7 legado

M7 publicado conta valores ausentes de `t1_planning_score`. A auditoria reproduz o cálculo por semestre e mostra que ele é uma transformação determinística de M6, não uma observação independente.

In [ ]:
legacy_m7 = pd.read_csv(M7_PATH, dtype={'Semestre':str})
signals = pd.read_csv(SIGNALS_PATH, dtype={'Semestre':str})
recalculated = (signals.assign(legacy_omitted=signals.t1_planning_score.isna())
    .groupby('Semestre', dropna=False)['legacy_omitted'].agg(n_teams='size', n_omitted='sum').reset_index())
recalculated['omission_rate'] = recalculated.n_omitted / recalculated.n_teams
recalculated = pd.concat([recalculated, pd.DataFrame([{'Semestre':'all','n_teams':len(signals),'n_omitted':int(signals.t1_planning_score.isna().sum()),'omission_rate':signals.t1_planning_score.isna().mean()}])], ignore_index=True)
comparison = legacy_m7.merge(recalculated, on='Semestre', suffixes=('_published','_recalculated'), validate='one_to_one')
assert np.allclose(comparison.omission_rate_published, comparison.omission_rate_recalculated)
assert (comparison.n_teams_published == comparison.n_teams_recalculated).all()
assert (comparison.n_omitted_published == comparison.n_omitted_recalculated).all()
comparison

## 3. Teste do construto

A alegação forte de M7 seria: ausência de commit T1 identifica omissão de planejamento. Esse teste pode ser falsificado se equipes sem commits possuírem artefatos T1 ou se forem observadas por avaliadores no marco T1. A segunda condição não prova planejamento, mas prova que inatividade de Git não é ausência de atividade do projeto.

In [ ]:
keys = ['ID_Equipe', 'Semestre']
planning = pd.read_parquet(PLANNING_PATH)
commits = pd.read_parquet(COMMITS_PATH)
evaluators = pd.read_parquet(EVALUATOR_PATH)
t1_commit_counts = commits.loc[commits.temporal_marker.eq('T1')].groupby(keys).size().rename('t1_commit_n').reset_index()
t1_evaluator_counts = evaluators.loc[evaluators.temporal_marker.eq('T1')].groupby(keys).size().rename('t1_evaluator_record_n').reset_index()
team_evidence = (signals[keys + ['t1_planning_score']].merge(planning[keys + ['pi_available','pi_file_count_t1','pi_line_delta_t1']], on=keys, validate='one_to_one').merge(t1_commit_counts, on=keys, how='left').merge(t1_evaluator_counts, on=keys, how='left').fillna({'t1_commit_n':0, 't1_evaluator_record_n':0}))
team_evidence['legacy_omitted'] = team_evidence.t1_planning_score.isna()
team_evidence['repository_inactive_pre_t1'] = team_evidence.t1_commit_n.eq(0)
team_evidence['planning_artifact_present_t1'] = team_evidence.pi_file_count_t1.gt(0)
team_evidence['evaluator_observed_t1'] = team_evidence.t1_evaluator_record_n.gt(0)
assert team_evidence.legacy_omitted.equals(team_evidence.repository_inactive_pre_t1)
assert team_evidence.legacy_omitted.equals(~team_evidence.planning_artifact_present_t1)
team_evidence.sort_values(['Semestre','ID_Equipe'])

## 4. M7a e M7b

M7a reporta inatividade de repositório pré-T1 por coorte. M7b mostra a cobertura de avaliação entre equipes inativas. Essas saídas tornam explícita a fronteira observacional: Git não permite afirmar que planejamento não ocorreu fora do repositório.

In [ ]:
m7_team = team_evidence[keys + ['repository_inactive_pre_t1','planning_artifact_present_t1','evaluator_observed_t1','t1_commit_n']].copy()
m7_cohort = (m7_team.groupby('Semestre', dropna=False).agg(team_n=('ID_Equipe','size'), inactive_repository_n=('repository_inactive_pre_t1','sum'), inactive_with_evaluator_t1_n=('evaluator_observed_t1', lambda values: int(values[m7_team.loc[values.index, 'repository_inactive_pre_t1']].sum()))).reset_index())
m7_cohort['pre_t1_repository_inactivity_rate'] = m7_cohort.inactive_repository_n / m7_cohort.team_n
m7_cohort['inactive_evaluator_coverage_rate'] = m7_cohort.inactive_with_evaluator_t1_n / m7_cohort.inactive_repository_n.replace(0, np.nan)
assert m7_cohort.loc[m7_cohort.Semestre.eq('2025.2'), 'pre_t1_repository_inactivity_rate'].iloc[0] == 5/9
assert m7_cohort.loc[m7_cohort.Semestre.eq('2026.1'), 'pre_t1_repository_inactivity_rate'].iloc[0] == 0
m7_cohort

## 5. Redundância, decisão e checklist

M7 legado é perfeitamente redundante com a ausência estrutural de M6a no conjunto observado. Portanto não deve entrar como preditor adicional em M9. M7a pode permanecer somente como diagnóstico de cobertura/reposítório; M7b deve acompanhar qualquer interpretação para impedir o salto de ‘sem Git’ para ‘sem planejamento’.

In [ ]:
m7_decision = pd.DataFrame([
    ('M7 legado', 'retire as planning-omission metric', 'exact duplicate of M6a absence in observed data; Git cannot observe off-repository planning'),
    ('M7a', 'retain as secondary diagnostic', 'pre-T1 repository inactivity rate, explicitly not planning omission'),
    ('M7b', 'retain as coverage qualifier', 'shows inactive Git teams were nevertheless evaluated at T1'),
    ('M9', 'do not include M7 separately', 'would double-count M6a absence and create a deterministic duplicate predictor'),
    ('paper claim', 'replace 55% planning omission', 'state 5/9 2025.2 teams had no observed pre-T1 repository activity'),
], columns=['component','decision','reason'])
assert team_evidence.legacy_omitted.equals(~team_evidence.planning_artifact_present_t1)
assert team_evidence.loc[team_evidence.repository_inactive_pre_t1, 'evaluator_observed_t1'].all()
m7_decision

In [ ]:
assert detected_rq == 'RQ3'
assert len(team_evidence) == 14
assert team_evidence.legacy_omitted.sum() == 5
assert team_evidence.legacy_omitted.equals(team_evidence.repository_inactive_pre_t1)
assert team_evidence.legacy_omitted.equals(~team_evidence.planning_artifact_present_t1)
assert CONFIG['inference'] == 'descriptive_only'
m7_evidence_manifest = {'metric':'M7','rq':detected_rq,'unit_of_analysis':CONFIG['unit_of_analysis'],'legacy_source':str(SIGNALS_PATH.relative_to(PROJECT_ROOT)),'commit_source':str(COMMITS_PATH.relative_to(PROJECT_ROOT)),'planning_source':str(PLANNING_PATH.relative_to(PROJECT_ROOT)),'evaluator_source':str(EVALUATOR_PATH.relative_to(PROJECT_ROOT)),'legacy_omitted_n':int(team_evidence.legacy_omitted.sum()),'recommended_interpretation':'pre_t1_repository_inactivity_not_planning_omission'}
m7_evidence_manifest

## 6. Trajetória de inatividade em janela móvel

Cada equipe é ancorada no último voto de avaliador do T3, como em M3. Para cada dia de $-63$ a $+7$ em relação a essa âncora, calcula-se uma janela retrospectiva de sete dias $[e-7\text{ dias}, e)$; a equipe é inativa quando essa janela não contém commits. Os checkpoints T1--T3 são mantidos como leituras pontuais pela mesma regra de âncora por voto.

Essa série distingue inatividade persistente, início tardio e ativação progressiva. Mede somente inatividade recente de repositório, não ausência de planejamento, esforço fora do Git ou qualidade do projeto.

In [ ]:
from pipeline_config import EVALUATOR_TEMPORAL_CUTS


def last_evaluator_vote_anchors(project_root: Path) -> pd.DataFrame:
    anchors = []
    for semester, checkpoint_ranges in EVALUATOR_TEMPORAL_CUTS.items():
        evaluator_votes = pd.read_csv(project_root / f"data/processed/forms/{semester}/avaliadores.csv")
        evaluator_votes["ID_Equipe"] = (
            evaluator_votes["To which group do these scores refer?"]
            .str.extract(r"Group\s+(\d+)", expand=False)
            .astype(int)
            .map(lambda value: f"TEAM_{value:02d}")
        )
        evaluator_votes["vote_at"] = pd.to_datetime(
            evaluator_votes["Timestamp"], format="mixed"
        ).dt.tz_localize("America/Fortaleza")
        for marker, (start_date, end_date) in checkpoint_ranges.items():
            in_checkpoint = evaluator_votes.loc[
                evaluator_votes["vote_at"].dt.date.between(
                    pd.Timestamp(start_date).date(), pd.Timestamp(end_date).date()
                )
            ]
            anchors.append(
                in_checkpoint.groupby("ID_Equipe", as_index=False)["vote_at"].max().assign(
                    Semestre=semester,
                    temporal_marker=marker,
                )
            )
    return pd.concat(anchors, ignore_index=True)


checkpoint_anchors = last_evaluator_vote_anchors(PROJECT_ROOT)
rolling_rows = []
for anchor in checkpoint_anchors.to_dict("records"):
    team_commits = commits.loc[
        commits["Semestre"].astype(str).eq(anchor["Semestre"])
        & commits["ID_Equipe"].eq(anchor["ID_Equipe"])
    ]
    window_end = anchor["vote_at"].tz_convert("UTC")
    window_start = window_end - pd.Timedelta(days=CONFIG["rolling_window_days"])
    recent_commits = team_commits.loc[
        team_commits["timestamp"].ge(window_start)
        & team_commits["timestamp"].lt(window_end)
    ]
    rolling_rows.append({
        "ID_Equipe": anchor["ID_Equipe"],
        "Semestre": anchor["Semestre"],
        "temporal_marker": anchor["temporal_marker"],
        "checkpoint_anchor": anchor["vote_at"],
        "window_start": window_start,
        "window_end": window_end,
        "commit_n_7d": int(len(recent_commits)),
        "repository_inactive_7d": not bool(len(recent_commits)),
    })
rolling_inactivity = pd.DataFrame(rolling_rows)
rolling_inactivity_cohort = (
    rolling_inactivity.groupby(["Semestre", "temporal_marker"], as_index=False)
    .agg(
        team_n=("ID_Equipe", "size"),
        inactive_team_n=("repository_inactive_7d", "sum"),
        recent_commit_n=("commit_n_7d", "sum"),
    )
)
rolling_inactivity_cohort["rolling_7d_repository_inactivity_rate"] = (
    rolling_inactivity_cohort["inactive_team_n"] / rolling_inactivity_cohort["team_n"]
)
assert len(rolling_inactivity) == 42
assert not rolling_inactivity.duplicated(["ID_Equipe", "Semestre", "temporal_marker"]).any()
assert rolling_inactivity_cohort.groupby("Semestre")["temporal_marker"].nunique().eq(3).all()
rolling_inactivity_cohort.sort_values(["Semestre", "temporal_marker"])

In [ ]:
t3_anchors = checkpoint_anchors.loc[
    checkpoint_anchors["temporal_marker"].eq("T3")
].copy()
rolling_daily_rows = []
for anchor in t3_anchors.to_dict("records"):
    team_commits = commits.loc[
        commits["Semestre"].astype(str).eq(anchor["Semestre"])
        & commits["ID_Equipe"].eq(anchor["ID_Equipe"])
    ]
    presentation_anchor = anchor["vote_at"].tz_convert("UTC")
    for day_offset in range(
        CONFIG["rolling_end_day_range_relative_to_t3"][0],
        CONFIG["rolling_end_day_range_relative_to_t3"][1] + 1,
        CONFIG["rolling_step_days"],
    ):
        window_end = presentation_anchor + pd.Timedelta(days=day_offset)
        window_start = window_end - pd.Timedelta(days=CONFIG["rolling_window_days"])
        recent_commits = team_commits.loc[
            team_commits["timestamp"].ge(window_start)
            & team_commits["timestamp"].lt(window_end)
        ]
        rolling_daily_rows.append({
            "ID_Equipe": anchor["ID_Equipe"],
            "Semestre": anchor["Semestre"],
            "window_end_day_relative_to_t3": day_offset,
            "commit_n_7d": int(len(recent_commits)),
            "repository_inactive_7d": not bool(len(recent_commits)),
        })
rolling_daily = pd.DataFrame(rolling_daily_rows)
rolling_daily_cohort = (
    rolling_daily.groupby(["Semestre", "window_end_day_relative_to_t3"], as_index=False)
    .agg(
        team_n=("ID_Equipe", "size"),
        inactive_team_n=("repository_inactive_7d", "sum"),
        recent_commit_n=("commit_n_7d", "sum"),
    )
)
rolling_daily_cohort["rolling_7d_repository_inactivity_rate"] = (
    rolling_daily_cohort["inactive_team_n"] / rolling_daily_cohort["team_n"]
)
assert len(rolling_daily) == 14 * 71
assert rolling_daily.groupby(["ID_Equipe", "Semestre"]).size().eq(71).all()
assert rolling_daily_cohort.groupby("Semestre").size().eq(71).all()
rolling_daily_cohort.loc[
    rolling_daily_cohort["window_end_day_relative_to_t3"].isin([-56, -49, -42, -21, -7, 0, 7])
].sort_values(["Semestre", "window_end_day_relative_to_t3"])

### 6.1 Padrões de persistência entre checkpoints

O resumo por checkpoints não substitui a série diária; ele apenas oferece uma classificação legível da inatividade observada nas três janelas de avaliação: nunca inativa, somente T1, intermitente ou persistente. Não há interpretação causal nem inferência sobre planejamento fora do Git.

In [ ]:
checkpoint_pattern = (
    rolling_inactivity.pivot_table(
        index=["ID_Equipe", "Semestre"],
        columns="temporal_marker",
        values="repository_inactive_7d",
        aggfunc="first",
    )
    .reindex(columns=["T1", "T2", "T3"])
    .reset_index()
)
checkpoint_pattern["inactive_checkpoint_n"] = checkpoint_pattern[["T1", "T2", "T3"]].sum(axis=1).astype(int)
checkpoint_pattern["inactivity_pattern"] = np.select(
    [
        checkpoint_pattern["inactive_checkpoint_n"].eq(0),
        checkpoint_pattern[["T1", "T2", "T3"]].eq([True, False, False]).all(axis=1),
        checkpoint_pattern["inactive_checkpoint_n"].eq(3),
    ],
    ["never_inactive", "t1_only", "persistent_all_checkpoints"],
    default="intermittent",
)
assert checkpoint_pattern["T3"].eq(False).all()
assert checkpoint_pattern["inactive_checkpoint_n"].between(0, 2).all()
checkpoint_pattern.sort_values(["Semestre", "ID_Equipe"])


In [ ]:
m7_temporal_decision = pd.DataFrame([
    ("M7 legado", "retire como planning-omission", "é duplicata determinística da ausência M6a e Git não observa planejamento externo"),
    ("M7a", "adote como saída principal", "trajetória diária de inatividade Git em janela retrospectiva de 7 dias, alinhada ao T3"),
    ("M7b", "adote como saída complementar", "padrão por checkpoints separa T1-only de inatividade intermitente"),
    ("M7c", "retenha como qualificador", "cobertura de avaliação T1 evita inferir ausência de atividade do projeto"),
    ("M9", "não incluir M7", "M7 é diagnóstico de dinâmica Git, não preditor independente de planejamento"),
], columns=["component", "decision", "reason"])
assert CONFIG["primary_outputs"][0] == "rolling_7d_repository_inactivity_trajectory"
assert checkpoint_pattern["inactivity_pattern"].notna().all()
m7_temporal_decision


In [ ]:
m7_temporal_evidence_manifest = {
    "metric": "M7",
    "rq": detected_rq,
    "analysis_level": CONFIG["unit_of_analysis"],
    "commit_source": str(COMMITS_PATH.relative_to(PROJECT_ROOT)),
    "evaluator_vote_source_pattern": "data/processed/forms/{semester}/avaliadores.csv",
    "checkpoint_anchor": CONFIG["checkpoint_anchor"],
    "rolling_alignment_anchor": CONFIG["rolling_alignment_anchor"],
    "rolling_window_days": CONFIG["rolling_window_days"],
    "rolling_step_days": CONFIG["rolling_step_days"],
    "rolling_end_day_range_relative_to_t3": CONFIG["rolling_end_day_range_relative_to_t3"],
    "team_semester_n": int(t3_anchors.shape[0]),
    "daily_windows_n": int(rolling_daily.shape[0]),
    "inference": CONFIG["inference"],
    "recommended_interpretation": "recent_repository_inactivity_not_planning_omission",
}
assert m7_temporal_evidence_manifest["daily_windows_n"] == 14 * 71
assert m7_temporal_evidence_manifest["recommended_interpretation"] == "recent_repository_inactivity_not_planning_omission"
m7_temporal_evidence_manifest